In [1]:
import heapq

GRAPH = {"S": [("A", 1), ("B", 4)], "A": [("C", 2)], "B": [("G", 5)],
         "C": [("G", 3)], "G": []}
H = {"S": 5, "A": 4, "B": 4, "C": 2, "G": 0}


def astar(graph, h, start, goal, verbose=False):
    """Returns (cost, path, expanded_nodes_list). cost=None if unreachable."""
    pq = [(h[start], 0, start, [start])]
    best = {start: 0}
    expanded, step = [], 0
    while pq:
        if verbose:
            print("   frontier:", sorted((f, g, n) for f, g, n, _ in pq))
        f, g, node, path = heapq.heappop(pq)
        if g != best[node]:                      # stale entry
            if verbose: print(f"   skip stale {node} (g={g})")
            continue
        if node == goal:                         # goal: logged, not counted
            if verbose: print(f"   GOAL popped: {node} g={g} h={h[node]} f={f}")
            return g, path, expanded
        step += 1
        expanded.append(node)
        if verbose:
            print(f"Expansion {step}: pop {node}  g={g} h={h[node]} f={g}+{h[node]}={f}")
        for nxt, w in graph.get(node, []):
            ng = g + w
            if ng < best.get(nxt, float("inf")):
                best[nxt] = ng
                heapq.heappush(pq, (ng + h[nxt], ng, nxt, path + [nxt]))
                if verbose:
                    print(f"     push {nxt}: g={g}+{w}={ng}, h={h[nxt]}, f={ng+h[nxt]}")
    return None, None, expanded


def true_costs(graph, goal):
    """Exact remaining cost to goal via Dijkstra on the reversed graph."""
    rev = {n: [] for n in graph}
    for u, es in graph.items():
        for v, w in es:
            rev[v].append((u, w))
    dist, pq = {goal: 0}, [(0, goal)]
    while pq:
        d, n = heapq.heappop(pq)
        if d > dist[n]: continue
        for p, w in rev[n]:
            if d + w < dist.get(p, 1e9):
                dist[p] = d + w; heapq.heappush(pq, (d + w, p))
    return dist


def audit(graph, h, goal):
    t = true_costs(graph, goal)
    print(f"{'node':<5}{'h':>4}{'true':>6}  admissible?")
    adm = True
    for n in graph:
        ok = h[n] <= t.get(n, float('inf')); adm &= ok
        print(f"{n:<5}{h[n]:>4}{t.get(n,'inf'):>6}  {ok}")
    cons = True
    for u, es in graph.items():
        for v, w in es:
            ok = h[u] <= w + h[v]; cons &= ok
            print(f"  edge {u}->{v}: h({u})={h[u]} <= {w}+h({v})={w+h[v]} ? {ok}")
    print("ADMISSIBLE:", adm, "| CONSISTENT:", cons)


if __name__ == "__main__":
    print("=== 1. Supplied code output ===")
    print(astar(GRAPH, H, "S", "G")[:2])

    print("\n=== 2. Heuristic audit (admissible + consistent) ===")
    audit(GRAPH, H, "G")

    print("\n=== 3. A* manual-trace check (verbose) ===")
    c, p, ex = astar(GRAPH, H, "S", "G", verbose=True)
    print("RESULT:", c, p, "| expanded:", ex, "| count:", len(ex))

    print("\n=== 4. UCS control (all h = 0) ===")
    H0 = {n: 0 for n in GRAPH}
    c0, p0, ex0 = astar(GRAPH, H0, "S", "G", verbose=True)
    print("RESULT:", c0, p0, "| expanded:", ex0, "| count:", len(ex0))

    print("\n=== 5. Overestimate: h(C) 2 -> 10 (true=3) ===")
    H1 = dict(H); H1["C"] = 10
    c1, p1, ex1 = astar(GRAPH, H1, "S", "G", verbose=True)
    print("RESULT:", c1, p1, "| expanded:", ex1, "| count:", len(ex1))

    print("\n=== 6. Overestimate that does NOT change answer: h(B) 4 -> 10 (true=5) ===")
    H2 = dict(H); H2["B"] = 10
    c2, p2, ex2 = astar(GRAPH, H2, "S", "G")
    print("RESULT:", c2, p2, "| expanded:", ex2, "| count:", len(ex2))

    print("\n=== 7. Edge cases ===")
    print("start == goal      :", astar(GRAPH, H, "G", "G")[:2])
    g2 = {"S": [("A", 1)], "A": [], "G": []}
    print("unreachable goal   :", astar(g2, H, "S", "G")[:2])
    g3 = {"S": [("A", 1), ("B", 1)], "A": [("G", 1)], "B": [("G", 1)], "G": []}
    print("tie (two cost-2)   :", astar(g3, {"S":0,"A":0,"B":0,"G":0}, "S", "G")[:2])

    print("\n=== 8. Summary table ===")
    print(f"{'run':<22}{'cost':>5}  {'path':<14}{'expanded'}")
    for name, r in [("A* (h supplied)", (c, p, ex)), ("UCS (h=0)", (c0, p0, ex0)),
                    ("A* h(C)=10", (c1, p1, ex1)), ("A* h(B)=10", (c2, p2, ex2))]:
        print(f"{name:<22}{r[0]:>5}  {'->'.join(r[1]):<14}{len(r[2])} {r[2]}")

=== 1. Supplied code output ===
(6, ['S', 'A', 'C', 'G'])

=== 2. Heuristic audit (admissible + consistent) ===
node    h  true  admissible?
S       5     6  True
A       4     5  True
B       4     5  True
C       2     3  True
G       0     0  True
  edge S->A: h(S)=5 <= 1+h(A)=5 ? True
  edge S->B: h(S)=5 <= 4+h(B)=8 ? True
  edge A->C: h(A)=4 <= 2+h(C)=4 ? True
  edge B->G: h(B)=4 <= 5+h(G)=5 ? True
  edge C->G: h(C)=2 <= 3+h(G)=3 ? True
ADMISSIBLE: True | CONSISTENT: True

=== 3. A* manual-trace check (verbose) ===
   frontier: [(5, 0, 'S')]
Expansion 1: pop S  g=0 h=5 f=0+5=5
     push A: g=0+1=1, h=4, f=5
     push B: g=0+4=4, h=4, f=8
   frontier: [(5, 1, 'A'), (8, 4, 'B')]
Expansion 2: pop A  g=1 h=4 f=1+4=5
     push C: g=1+2=3, h=2, f=5
   frontier: [(5, 3, 'C'), (8, 4, 'B')]
Expansion 3: pop C  g=3 h=2 f=3+2=5
     push G: g=3+3=6, h=0, f=6
   frontier: [(6, 6, 'G'), (8, 4, 'B')]
   GOAL popped: G g=6 h=0 f=6
RESULT: 6 ['S', 'A', 'C', 'G'] | expanded: ['S', 'A', 'C'] | coun